Tehran housing-price forecasting: reproducibility workflow

This notebook rebuilds the analytical panel, validates temporal information boundaries, estimates the prespecified Ridge models, evaluates the untouched chronological test period, and exports the tables and figures used for review. Run the cells from top to bottom.

The notebook uses the aggregate district-month workbook supplied with this repository. The four environmental-oriented proxies are heuristic market-level indices; they are not measured building energy use, carbon emissions, or LCA scores.

Stage 1 — Load and validate the source workbook

This stage reads `Sheet2`, checks the 22-district by 89-month panel, corrects the documented factor-of-100 price-scale error, sets inconsistent early USD observations to missing, and records retrospective decimal-shift flags. The centred-window flags are restricted to months after the analytical sample and are not used to construct model predictors.

In [ ]:
"""Stage 1: load, validate, and audit the workbook."""
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from tehran_housing.pipeline import (
    SEED, load_study_data, build_analysis_frame, run_rolling_validation,
    run_final_models, paired_bootstrap, run_month_shuffle_permutation,
    run_lodo, run_scenario_sensitivity, run_regional_scenario_sensitivity,
    run_nonlinear_models, run_shap_attribution, run_residual_diagnostics,
    _save_figures, persistence_predictions, FIRST_ANALYSIS_MONTH, TRAIN_END_MONTH,
    TEST_START_MONTH, LAST_ANALYSIS_MONTH,
)

DATA_PATH = ROOT / "data" / "raw" / "Home_4.xlsx"
OUTPUT = ROOT / "outputs"
TABLES = OUTPUT / "tables"
OUTPUT.mkdir(exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

study = load_study_data(DATA_PATH)
with open(OUTPUT / "data_audit.json", "w", encoding="utf-8") as stream:
    json.dump(study.audit, stream, indent=2)
study.decimal_flags.to_csv(TABLES / "decimal_shift_diagnostics.csv", index=False)
study.audit


Stage 2 — Build the lagged analytical panel and environmental proxies

The target is one-month log-price growth. Every autoregressive and macroeconomic predictor is built from information available no later than month t−1. The previous observed log price is stored as a separate forecast anchor; it is not inferred by subtracting the target from the current outcome.

In [ ]:
"""Stage 2: construct lag-only features, seasonal controls, and proxies."""
analysis = build_analysis_frame(study)
rows = analysis.rows
specifications = analysis.specifications
rows.to_csv(OUTPUT / "analysis_panel.csv", index=False)
proxy_summary = pd.DataFrame([
    {"proxy": col,
     "mean": rows.drop_duplicates("t")[col].mean(),
     "sd": rows.drop_duplicates("t")[col].std(ddof=1),
     "min": rows.drop_duplicates("t")[col].min(),
     "median": rows.drop_duplicates("t")[col].median(),
     "max": rows.drop_duplicates("t")[col].max()}
    for col in ["ECI", "CCF", "HED", "SDI"]
])
proxy_summary.to_csv(TABLES / "proxy_descriptive_statistics.csv", index=False)
print(f"Analytical rows: {len(rows):,}")
print(f"Training rows: {((rows.t >= FIRST_ANALYSIS_MONTH) & (rows.t <= TRAIN_END_MONTH)).sum():,}")
print(f"Untouched test rows: {((rows.t >= TEST_START_MONTH) & (rows.t <= LAST_ANALYSIS_MONTH)).sum():,}")
proxy_summary


Stage 3 — Nested rolling-origin validation

Five six-month outer validation windows are evaluated in chronological order. For each outer fold, Ridge alpha is selected on three trailing, non-overlapping three-month inner validation blocks. Robust scaling is re-estimated inside each inner training window.

In [ ]:
"""Stage 3: estimate outer rolling-origin validation results."""
cv = run_rolling_validation(rows, specifications)
cv.to_csv(TABLES / "rolling_origin_validation.csv", index=False)
cv.groupby("specification")["RMSE"].agg(["mean", "std"])


Stage 4 — Fit final models and evaluate the untouched test period

The final models are trained on t=13–63 and evaluated on t=64–76. Price-level predictions use the directly observed t−1 price as the anchor. The rank-adjusted controlled specifications remove exact linear duplicates and retain CCF as the only proxy feature that adds a distinct linear dimension to the macro-control design.

In [ ]:
"""Stage 4: fit final Ridge specifications and save test predictions."""
models = run_final_models(rows, specifications)
months = rows["t"].to_numpy(dtype=int)
test_mask = (months >= TEST_START_MONTH) & (months <= LAST_ANALYSIS_MONTH)
metric_rows = []
predictions = {}
for name, fitted in models.items():
    metric_rows.append({"specification": name,
                        "n_features": len(fitted.feature_names),
                        "alpha": fitted.alpha,
                        **fitted.metrics})
    fitted.alpha_profile.to_csv(TABLES / f"alpha_profile_{name}.csv", index=False)
    fitted.predictions.to_csv(TABLES / f"test_predictions_{name}.csv", index=False)
    predictions[name] = fitted.predictions["predicted_price"].to_numpy(dtype=float)
persistence_metric, persistence_pred = persistence_predictions(rows, test_mask)
metric_rows.append({"specification": "persistence", "n_features": 0,
                    "alpha": np.nan, **persistence_metric})
persistence_pred.to_csv(TABLES / "test_predictions_persistence.csv", index=False)
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(TABLES / "out_of_time_test_metrics.csv", index=False)
metrics


Stage 5 — Quantify uncertainty and run the month-shuffle diagnostic

Paired bootstrap resamples use identical sampled observations for both models. The three-month block bootstrap keeps all 22 district observations together within each selected month. The month-shuffle permutation is a diagnostic comparing the original four-proxy specification against the baseline; it does not test incremental proxy information beyond macro-ratio controls.

In [ ]:
"""Stage 5: paired bootstrap comparisons and joint month-shuffle permutation."""
test_rows = rows.loc[test_mask]
actual = test_rows["price"].to_numpy(dtype=float)
test_months = test_rows["t"].to_numpy(dtype=int)
districts = test_rows["district"].to_numpy(dtype=int)
comparisons = [
    ("full_proxy_vs_baseline", "baseline", "full_proxy"),
    ("full_proxy_vs_macro_control", "macro_control", "full_proxy"),
    ("rank_ccf_vs_rank_control", "rank_macro_control", "rank_macro_plus_ccf"),
]
bootstrap_records = []
for offset, (label, reference, comparison) in enumerate(comparisons):
    summary = paired_bootstrap(actual, predictions[reference], predictions[comparison],
                               test_months, districts, reps=2000, seed=SEED + offset)
    for scheme, values in summary.items():
        bootstrap_records.append({"comparison": label, "bootstrap_scheme": scheme, **values})
bootstrap = pd.DataFrame(bootstrap_records)
bootstrap.to_csv(TABLES / "paired_bootstrap_comparisons.csv", index=False)

permutation = run_month_shuffle_permutation(
    rows, models["baseline"], models["full_proxy"],
    specifications["baseline"], specifications["full_proxy"],
    reps=500, seed=SEED,
)
permutation.to_csv(TABLES / "month_shuffle_permutation.csv", index=False)
bootstrap


Stage 6 — District-omission and regional scenario analyses

The district-omission analysis removes each district from training and scores the complete test panel. This is a sensitivity-to-training-composition diagnostic, not a held-out-district-only performance estimate. Regional scenarios use price terciles defined from mean district log prices in the training period and report model-based prediction sensitivities.

In [ ]:
"""Stage 6: estimate district-omission and price-tercile scenario results."""
lodo = run_lodo(rows, specifications)
lodo.to_csv(TABLES / "leave_one_district_out.csv", index=False)

scenarios = run_scenario_sensitivity(rows, models["full_proxy"],
                                     specifications["full_proxy"], reps=300)
scenarios.to_csv(TABLES / "scenario_sensitivity.csv", index=False)

regional, tier_membership, tier_model_metrics = run_regional_scenario_sensitivity(
    rows, specifications, reps=300
)
regional.to_csv(TABLES / "regional_scenario_sensitivity.csv", index=False)
tier_membership.to_csv(TABLES / "district_price_tiers.csv", index=False)
pd.DataFrame([tier_model_metrics]).to_csv(TABLES / "tier_interaction_model_metrics.csv", index=False)
regional.query("parameter_change_pct == 30")


Stage 7 — Nonlinear comparators, SHAP attribution, and residual diagnostics

Random Forest and XGBoost are fitted with the prespecified fixed hyperparameters; they are not presented as tuned state-of-the-art searches. SHAP is used as a descriptive model-attribution tool, not as a causal or independent-information estimator. Residual diagnostics are reported descriptively because the test sample contains only 13 common time points.

In [ ]:
"""Stage 7: fit nonlinear comparators, compute SHAP summaries, and diagnose residuals."""
nonlinear = run_nonlinear_models(rows, specifications)
nonlinear.to_csv(TABLES / "nonlinear_model_comparison.csv", index=False)
shap_importance, shap_groups = run_shap_attribution(rows, specifications, models["full_proxy"])
shap_importance.to_csv(TABLES / "shap_feature_attribution.csv", index=False)
shap_groups.to_csv(TABLES / "shap_group_attribution.csv", index=False)
residual_diagnostics = run_residual_diagnostics(models["rank_macro_plus_ccf"].predictions)
with open(OUTPUT / "residual_diagnostics.json", "w", encoding="utf-8") as stream:
    json.dump(residual_diagnostics, stream, indent=2)
nonlinear


Stage 8 — Export figures, summary tables, and a reproducibility manifest

The final stage writes the figures, run summary, and a manifest containing the input workbook hash and software versions. This allows reviewers to identify the exact input and execution environment used for the run. Re-run the notebook after changing any preprocessing or model specification so the exported artefacts remain consistent.

In [ ]:
"""Stage 8: write the reproducibility figures and run summary."""
_save_figures(OUTPUT, rows, cv, models, bootstrap, lodo, scenarios,
              regional, residual_diagnostics, shap_groups)
summary = {
    "seed": SEED,
    "analysis_rows": int(len(rows)),
    "train_rows": int(((rows.t >= FIRST_ANALYSIS_MONTH) & (rows.t <= TRAIN_END_MONTH)).sum()),
    "test_rows": int(test_mask.sum()),
    "models": metrics.to_dict(orient="records"),
    "permutation_test": permutation.to_dict(orient="records"),
    "residual_diagnostics": residual_diagnostics,
}
with open(OUTPUT / "run_summary.json", "w", encoding="utf-8") as stream:
    json.dump(summary, stream, indent=2)
print(f"Tables: {TABLES.resolve()}")
print(f"Figures: {(OUTPUT / 'figures').resolve()}")

import hashlib
import importlib.metadata as metadata
import platform

package_names = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
                 "openpyxl", "statsmodels", "xgboost", "shap"]
manifest = {
    "python": platform.python_version(),
    "seed": SEED,
    "input_file": str(DATA_PATH.relative_to(ROOT)),
    "input_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "packages": {name: metadata.version(name) for name in package_names},
}
with open(OUTPUT / "reproducibility_manifest.json", "w", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2)
